<a href="https://colab.research.google.com/github/Raulfb04/Simulaci-n-1/blob/main/Sheldon_Ross.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema de colas M/M/1

Se simula un sistema con:

- Llegadas Poisson con tasa $\lambda$
- Tiempos de servicio exponenciales con tasa $\mu$
- Un solo servidor
- Disciplina FIFO

Datos de prueba:

$\lambda = 4$, $\mu = 6$

In [1]:
import numpy as np
import pandas as pd

# Parámetros
lam = 4
mu = 6
N = 100000

# Solución analítica M/M/1

rho = lam / mu
P0 = 1 - rho
Lq = lam**2 / (mu * (mu - lam))
Ls = Lq + rho
Wq = Lq / lam
Ws = Ls / lam

analitico = pd.DataFrame({
    "Medida": ["rho", "P0", "Lq", "Ls", "Wq", "Ws"],
    "Valor analítico": [rho, P0, Lq, Ls, Wq, Ws]
})

analitico

,Medida,Valor analítico
0,rho,0.666667
1,P0,0.333333
2,Lq,1.333333
3,Ls,2.000000
4,Wq,0.333333
5,Ws,0.500000


## Simulación

El seudocódigo de Sheldon Ross se basa en generar tiempos entre llegadas y tiempos de servicio mediante variables exponenciales.

Para cada cliente:

1. Se genera su tiempo de llegada.
2. Se genera su tiempo de servicio.
3. Si el servidor está libre, inicia servicio inmediatamente.
4. Si el servidor está ocupado, espera en cola.
5. Se calculan tiempos de espera, salida y permanencia en el sistema.

In [2]:
np.random.seed(123)

# Tiempos entre llegadas y tiempos de servicio
tiempos_entre_llegadas = np.random.exponential(1/lam, N)
tiempos_servicio = np.random.exponential(1/mu, N)

# Tiempos de llegada acumulados
llegadas = np.cumsum(tiempos_entre_llegadas)

inicio_servicio = np.zeros(N)
salidas = np.zeros(N)
espera_cola = np.zeros(N)
tiempo_sistema = np.zeros(N)

for i in range(N):
    if i == 0:
        inicio_servicio[i] = llegadas[i]
    else:
        inicio_servicio[i] = max(llegadas[i], salidas[i-1])

    salidas[i] = inicio_servicio[i] + tiempos_servicio[i]
    espera_cola[i] = inicio_servicio[i] - llegadas[i]
    tiempo_sistema[i] = salidas[i] - llegadas[i]

# Estimaciones por simulación
Wq_sim = np.mean(espera_cola)
Ws_sim = np.mean(tiempo_sistema)

Lq_sim = lam * Wq_sim
Ls_sim = lam * Ws_sim

rho_sim = np.sum(tiempos_servicio) / salidas[-1]
P0_sim = 1 - rho_sim

simulacion = pd.DataFrame({
    "Medida": ["rho", "P0", "Lq", "Ls", "Wq", "Ws"],
    "Valor simulación": [rho_sim, P0_sim, Lq_sim, Ls_sim, Wq_sim, Ws_sim]
})

simulacion

,Medida,Valor simulación
0,rho,0.667550
1,P0,0.332450
2,Lq,1.312440
3,Ls,1.980113
4,Wq,0.328110
5,Ws,0.495028


In [3]:
comparacion = analitico.copy()
comparacion["Valor simulación"] = simulacion["Valor simulación"]
comparacion["Error absoluto"] = abs(
    comparacion["Valor analítico"] - comparacion["Valor simulación"]
)

comparacion

,Medida,Valor analítico,Valor simulación,Error absoluto
0,rho,0.666667,0.667550,0.000883
1,P0,0.333333,0.332450,0.000883
2,Lq,1.333333,1.312440,0.020893
3,Ls,2.000000,1.980113,0.019887
4,Wq,0.333333,0.328110,0.005223
5,Ws,0.500000,0.495028,0.004972


In [4]:
analitico

,Medida,Valor analítico
0,rho,0.666667
1,P0,0.333333
2,Lq,1.333333
3,Ls,2.000000
4,Wq,0.333333
5,Ws,0.500000
